# TEMPO Foundation Model Analysis

**Purpose:** Comprehensive analysis of TEMPO zero-shot and fine-tuned performance for carbon flux prediction

**Author:** Paul Ezennolim  
**Date:** February 2025  
**Thesis:** Temporal-spatial data fusion for carbon flux quatification in agroecosystems: a multimodal learning approach

---

This notebook analyzes:
1. TEMPO zero-shot performance (pre-trained model, no domain adaptation)
2. TEMPO fine-tuned performance (adapted to carbon flux)
3. Comparison with baseline models (RF, XGBoost, LSTM)
4. Knowledge-guided decomposition analysis (STL components)
5. Cross-ecosystem generalization patterns

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

# Paths
RESULTS_DIR = Path('../results')
METRICS_DIR = RESULTS_DIR / 'metrics'
PREDS_DIR = RESULTS_DIR / 'predictions'
DATA_DIR = Path('../data/processed')
FIG_DIR = Path('../thesis/figures')
FIG_DIR.mkdir(exist_ok=True, parents=True)

SITES = ['UK-AMo', 'SE-Htm']
SITE_INFO = {
    'UK-AMo': {'name': 'Auchencorth Moss', 'ecosystem': 'Peatland', 'country': 'UK'},
    'SE-Htm': {'name': 'Hyltemossa', 'ecosystem': 'Forest', 'country': 'Sweden'},
}

# Consistent colours across all figures
MODEL_COLORS = {
    'Random Forest': '#3498db',
    'XGBoost': '#2ecc71',
    'LSTM': '#e74c3c',
    'TEMPO Zero-Shot': '#f39c12',
    'TEMPO Fine-Tuned': '#9b59b6',
}
MODEL_ORDER = list(MODEL_COLORS.keys())

print('Setup complete.')

## 1. Load All Model Results

Loading metrics from baselines and TEMPO variants for comprehensive comparison.

In [ ]:
# Mapping from display name -> metrics JSON key / filename stem
MODEL_FILES = {
    'Random Forest': 'randomforest',
    'XGBoost': 'xgboost',
    'LSTM': 'lstm',
    'TEMPO Zero-Shot': 'tempo_zero_shot',
    'TEMPO Fine-Tuned': 'tempo_fine_tuned',
}

all_metrics = {}
for display_name, file_key in MODEL_FILES.items():
    path = METRICS_DIR / f'{file_key}_metrics.json'
    if path.exists():
        with open(path) as f:
            all_metrics[display_name] = json.load(f)
        print(f'Loaded {display_name}')
    else:
        print(f'  {path.name} not found')

# Build comparison DataFrame, skipping NaN entries
rows = []
for model_name, site_metrics in all_metrics.items():
    for site in SITES:
        if site not in site_metrics:
            continue
        m = site_metrics[site]
        rmse = m.get('RMSE')
        mae = m.get('MAE')
        r2 = m.get('R2', m.get('R\u00b2'))
        if any(v is None or (isinstance(v, float) and np.isnan(v))
               for v in [rmse, mae, r2]):
            print(f'  Skipping {model_name} / {site} (NaN)')
            continue
        rows.append({
            'Model': model_name, 'Site': site,
            'RMSE': rmse, 'MAE': mae, 'R\u00b2': r2,
        })

comparison_df = pd.DataFrame(rows)
valid_models = [m for m in MODEL_ORDER if m in comparison_df['Model'].values]

print(f'\nModels with valid metrics: {valid_models}\n')
display(
    comparison_df.pivot_table(
        index='Model', columns='Site',
        values=['RMSE', 'MAE', 'R\u00b2'],
    ).round(4).reindex([m for m in MODEL_ORDER if m in comparison_df['Model'].values])
)

## 2. TEMPO vs Baselines: Full Comparison

Side-by-side comparison of all models across both test sites.

In [ ]:
metrics_list = ['RMSE', 'MAE', 'R\u00b2']
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for row_idx, site in enumerate(SITES):
    site_df = comparison_df[comparison_df['Site'] == site].copy()
    # Enforce model ordering
    site_df['Model'] = pd.Categorical(
        site_df['Model'], categories=MODEL_ORDER, ordered=True
    )
    site_df = site_df.sort_values('Model').reset_index(drop=True)

    for col_idx, metric in enumerate(metrics_list):
        ax = axes[row_idx, col_idx]
        models = site_df['Model'].values
        values = site_df[metric].values
        colors = [MODEL_COLORS.get(m, 'gray') for m in models]

        bars = ax.bar(range(len(models)), values, color=colors, alpha=0.85)

        # Highlight TEMPO bars with border
        for i, m in enumerate(models):
            if 'TEMPO' in m:
                bars[i].set_edgecolor('black')
                bars[i].set_linewidth(2)

        # Value labels
        for bar, v in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2, bar.get_height(),
                f'{v:.3f}', ha='center', va='bottom',
                fontsize=8, fontweight='bold',
            )

        short = [m.replace('TEMPO ', 'T-').replace('Random Forest', 'RF')
                 for m in models]
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(short, rotation=45, ha='right', fontsize=9)
        ax.set_ylabel(metric, fontsize=11, fontweight='bold')
        ax.set_title(
            f"{site} ({SITE_INFO[site]['ecosystem']})",
            fontsize=12, fontweight='bold',
        )
        ax.grid(True, alpha=0.3, axis='y')

plt.suptitle(
    'Cross-Site Model Performance Comparison',
    fontsize=15, fontweight='bold', y=1.0,
)
plt.tight_layout()
plt.savefig(FIG_DIR / 'all_models_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'all_models_comparison.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: all_models_comparison.png')

## 3. Fine-Tuning Impact Analysis

Quantifying the improvement (or degradation) from fine-tuning TEMPO on carbon flux data.

In [ ]:
# Compute per-site improvement
ft_rows = []
for site in SITES:
    zs = comparison_df[
        (comparison_df['Model'] == 'TEMPO Zero-Shot') &
        (comparison_df['Site'] == site)
    ]
    ft = comparison_df[
        (comparison_df['Model'] == 'TEMPO Fine-Tuned') &
        (comparison_df['Site'] == site)
    ]
    if zs.empty or ft.empty:
        continue

    zs_rmse, ft_rmse = zs['RMSE'].values[0], ft['RMSE'].values[0]
    zs_r2, ft_r2 = zs['R\u00b2'].values[0], ft['R\u00b2'].values[0]

    ft_rows.append({
        'Site': site,
        'Ecosystem': SITE_INFO[site]['ecosystem'],
        'ZS RMSE': zs_rmse,
        'FT RMSE': ft_rmse,
        'RMSE \u0394 (%)': (zs_rmse - ft_rmse) / zs_rmse * 100,
        'ZS R\u00b2': zs_r2,
        'FT R\u00b2': ft_r2,
        'R\u00b2 \u0394': ft_r2 - zs_r2,
    })

ft_df = pd.DataFrame(ft_rows)
print('=== Fine-Tuning Impact ===\n')
display(ft_df)

In [ ]:
if not ft_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    x = np.arange(len(ft_df))
    w = 0.35

    # --- RMSE panel ---
    ax = axes[0]
    ax.bar(x - w / 2, ft_df['ZS RMSE'], w,
           label='Zero-Shot', color=MODEL_COLORS['TEMPO Zero-Shot'], alpha=0.85)
    ax.bar(x + w / 2, ft_df['FT RMSE'], w,
           label='Fine-Tuned', color=MODEL_COLORS['TEMPO Fine-Tuned'], alpha=0.85)

    for i, row in ft_df.iterrows():
        pct = row['RMSE \u0394 (%)']
        y_pos = max(row['ZS RMSE'], row['FT RMSE']) + 0.08
        ax.text(i, y_pos, f"{pct:+.1f}%", ha='center', fontsize=10,
                fontweight='bold', color='green' if pct > 0 else 'red')

    ax.set_ylabel('RMSE (\u03bcmol CO\u2082 m\u207b\u00b2 s\u207b\u00b9)',
                   fontsize=11, fontweight='bold')
    ax.set_title('RMSE: Zero-Shot vs Fine-Tuned', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(ft_df['Site'])
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

    # --- R\u00b2 panel ---
    ax = axes[1]
    ax.bar(x - w / 2, ft_df['ZS R\u00b2'], w,
           label='Zero-Shot', color=MODEL_COLORS['TEMPO Zero-Shot'], alpha=0.85)
    ax.bar(x + w / 2, ft_df['FT R\u00b2'], w,
           label='Fine-Tuned', color=MODEL_COLORS['TEMPO Fine-Tuned'], alpha=0.85)

    for i, row in ft_df.iterrows():
        delta = row['R\u00b2 \u0394']
        y_pos = max(row['ZS R\u00b2'], row['FT R\u00b2']) + 0.02
        ax.text(i, y_pos, f"{delta:+.3f}", ha='center', fontsize=10,
                fontweight='bold', color='green' if delta > 0 else 'red')

    ax.set_ylabel('R\u00b2 Score', fontsize=11, fontweight='bold')
    ax.set_title('R\u00b2: Zero-Shot vs Fine-Tuned', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(ft_df['Site'])
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3, axis='y')

    plt.suptitle('TEMPO Fine-Tuning Impact', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'tempo_finetuning_impact.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIG_DIR / 'tempo_finetuning_impact.pdf', bbox_inches='tight')
    plt.show()
    print('Figure saved: tempo_finetuning_impact.png')

## 4. Best Model Identification

Ranking all models per site and determining where TEMPO places.

In [ ]:
for site in SITES:
    site_df = comparison_df[comparison_df['Site'] == site].copy()
    site_df = site_df.sort_values('RMSE')

    print(f'\n--- {site} ({SITE_INFO[site]["ecosystem"]}) ---')
    print(f'{"Rank":<5} {"Model":<22} {"RMSE":>8} {"MAE":>8} {"R\u00b2":>8}')
    print('-' * 55)
    for rank, (_, r) in enumerate(site_df.iterrows(), 1):
        marker = '  <--' if 'TEMPO' in r['Model'] else ''
        print(f"{rank:<5} {r['Model']:<22} {r['RMSE']:>8.4f} {r['MAE']:>8.4f} {r['R\u00b2']:>8.4f}{marker}")

## 5. Load Predictions and Targets

Loading prediction arrays for detailed forecast and error analysis.

In [ ]:
test_targets = {}
for site in SITES:
    test_targets[site] = np.load(DATA_DIR / f'test_{site}_y.npy')
    print(f'Targets  {site}: {test_targets[site].shape}')

# Load all available prediction files (skip those with NaN)
predictions = {}
for display_name, file_key in MODEL_FILES.items():
    predictions[display_name] = {}
    for site in SITES:
        for parent in [PREDS_DIR, PREDS_DIR / 'baselines']:
            path = parent / f'{file_key}_preds_{site}.npy'
            if path.exists():
                arr = np.load(path)
                if np.isnan(arr).any():
                    print(f'  {display_name:20s} / {site}: NaN, skipped')
                else:
                    predictions[display_name][site] = arr
                    print(f'  {display_name:20s} / {site}: {arr.shape}')
                break

plot_models = [m for m in MODEL_ORDER if any(predictions.get(m, {}).values())]
print(f'\nModels with valid predictions: {plot_models}')

## 6. TEMPO Forecast Examples

Comparing zero-shot and fine-tuned TEMPO against actual NEE for sample 96-hour windows.

In [ ]:
n_samples = 3
fig, axes = plt.subplots(
    len(SITES), n_samples,
    figsize=(6 * n_samples, 5 * len(SITES)),
    sharey='row',
)

for row, site in enumerate(SITES):
    n_seqs = len(test_targets[site])
    idxs = np.linspace(50, n_seqs - 50, n_samples, dtype=int)

    for col, seq_idx in enumerate(idxs):
        ax = axes[row, col]
        actual = test_targets[site][seq_idx]
        hours = np.arange(len(actual))

        ax.plot(hours, actual, 'k-', lw=2.5, label='Actual', alpha=0.9)

        for model in ['TEMPO Zero-Shot', 'TEMPO Fine-Tuned']:
            if site in predictions.get(model, {}):
                pred = predictions[model][site][seq_idx]
                ax.plot(
                    hours, pred, '--', lw=1.8,
                    label=model, color=MODEL_COLORS[model], alpha=0.8,
                )

        ax.axhline(0, color='red', ls=':', lw=0.8, alpha=0.5)
        ax.set_xlabel('Forecast Hour', fontsize=10)
        if col == 0:
            ax.set_ylabel(
                f"{site}\nNEE (\u03bcmol CO\u2082 m\u207b\u00b2 s\u207b\u00b9)",
                fontsize=10,
            )
        ax.set_title(f'Sequence {seq_idx}', fontsize=10)
        ax.grid(True, alpha=0.3)
        if row == 0 and col == n_samples - 1:
            ax.legend(loc='upper right', fontsize=8)

plt.suptitle(
    'TEMPO 96-hour Forecast Examples',
    fontsize=14, fontweight='bold', y=1.01,
)
plt.tight_layout()
plt.savefig(FIG_DIR / 'tempo_forecast_examples.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'tempo_forecast_examples.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: tempo_forecast_examples.png')

## 7. Actual vs Predicted Scatter (TEMPO variants)

Scatter plots showing prediction quality for zero-shot and fine-tuned TEMPO.

In [ ]:
tempo_models = [m for m in ['TEMPO Zero-Shot', 'TEMPO Fine-Tuned'] if m in plot_models]
n_tm = len(tempo_models)

if n_tm > 0:
    fig, axes = plt.subplots(
        len(SITES), n_tm, figsize=(6.5 * n_tm, 5.5 * len(SITES)),
        squeeze=False,
    )
    np.random.seed(42)

    for row, site in enumerate(SITES):
        actual_flat = test_targets[site].flatten()
        for col, model in enumerate(tempo_models):
            ax = axes[row, col]
            if site not in predictions.get(model, {}):
                ax.text(0.5, 0.5, 'No data', ha='center', va='center',
                        transform=ax.transAxes, fontsize=12)
                ax.set_title(f'{site} - {model}', fontsize=11)
                continue

            pred_flat = predictions[model][site].flatten()
            idx = np.random.choice(len(actual_flat), min(5000, len(actual_flat)), replace=False)

            ax.scatter(
                actual_flat[idx], pred_flat[idx],
                alpha=0.2, s=8, color=MODEL_COLORS[model],
            )
            lo = min(actual_flat.min(), pred_flat.min())
            hi = max(actual_flat.max(), pred_flat.max())
            ax.plot([lo, hi], [lo, hi], 'r--', lw=2, label='y = x')

            ss_res = np.sum((actual_flat - pred_flat) ** 2)
            ss_tot = np.sum((actual_flat - actual_flat.mean()) ** 2)
            r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

            ax.text(
                0.05, 0.95, f'R\u00b2 = {r2:.4f}',
                transform=ax.transAxes, va='top', fontsize=11, fontweight='bold',
                bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
            )
            ax.set_title(f'{site} - {model}', fontsize=11, fontweight='bold')
            ax.set_xlabel('Actual NEE', fontsize=10)
            ax.set_ylabel('Predicted NEE', fontsize=10)
            ax.legend(fontsize=8, loc='lower right')
            ax.grid(True, alpha=0.3)
            ax.set_aspect('equal', adjustable='datalim')

    plt.suptitle('Actual vs Predicted NEE (TEMPO)', fontsize=14, fontweight='bold', y=1.0)
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'tempo_scatter_plots.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIG_DIR / 'tempo_scatter_plots.pdf', bbox_inches='tight')
    plt.show()
    print('Figure saved: tempo_scatter_plots.png')

## 8. Horizon-wise Performance: All Models

How does prediction accuracy degrade across the 96-hour horizon for each model?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

all_hours = np.arange(1, 97)  # 1 to 96

for ax_idx, site in enumerate(SITES):
    ax = axes[ax_idx]
    actual = test_targets[site]

    for model in plot_models:
        if site not in predictions.get(model, {}):
            continue
        pred = predictions[model][site]
        rmse_curve = np.array([
            np.sqrt(np.mean((actual[:, h] - pred[:, h]) ** 2))
            for h in range(96)
        ])
        ax.plot(
            all_hours, rmse_curve, lw=1.5,
            label=model, color=MODEL_COLORS[model], alpha=0.85,
        )

    ax.set_xlabel('Forecast Hour', fontsize=11, fontweight='bold')
    ax.set_ylabel('RMSE', fontsize=11, fontweight='bold')
    ax.set_title(f"{site} ({SITE_INFO[site]['ecosystem']})",
                  fontsize=12, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle('RMSE Across Forecast Horizon (All Models)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'all_models_horizon_rmse.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'all_models_horizon_rmse.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: all_models_horizon_rmse.png')

## 9. Error Distribution Comparison

Overlayed error histograms for TEMPO variants vs best baseline.

In [ ]:
overlay_models = [m for m in ['XGBoost', 'TEMPO Zero-Shot', 'TEMPO Fine-Tuned']
                  if m in plot_models]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax_idx, site in enumerate(SITES):
    ax = axes[ax_idx]
    actual_flat = test_targets[site].flatten()

    for model in overlay_models:
        if site not in predictions.get(model, {}):
            continue
        pred_flat = predictions[model][site].flatten()
        errors = pred_flat - actual_flat

        ax.hist(
            errors, bins=80, density=True, alpha=0.45,
            color=MODEL_COLORS[model], label=model, edgecolor='none',
        )

    ax.axvline(0, color='black', ls='--', lw=1.5)
    ax.set_xlabel('Prediction Error', fontsize=11)
    ax.set_ylabel('Density', fontsize=11)
    ax.set_title(f"{site} ({SITE_INFO[site]['ecosystem']})",
                  fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Prediction Error Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIG_DIR / 'tempo_error_distributions.png', dpi=300, bbox_inches='tight')
plt.savefig(FIG_DIR / 'tempo_error_distributions.pdf', bbox_inches='tight')
plt.show()
print('Figure saved: tempo_error_distributions.png')

## 10. Knowledge-Guided Decomposition Analysis

Analyzing how well TEMPO captures the STL decomposition components (trend, seasonal, residual) of the NEE signal. This validates whether the foundation model implicitly learns physically-meaningful time series structure.

In [ ]:
decomp_path = METRICS_DIR / 'decomposition_metrics.json'
if decomp_path.exists():
    with open(decomp_path) as f:
        decomp_metrics = json.load(f)
    print('Loaded decomposition metrics')
    print(f'Sites with decomposition data: {list(decomp_metrics.keys())}')
else:
    decomp_metrics = None
    print('Decomposition metrics not found - run analyze_kgml_decomposition.py first')

In [ ]:
if decomp_metrics:
    # Collect decomposition summary across all available sites
    decomp_rows = []
    for site in SITES:
        if site not in decomp_metrics:
            continue
        entry = decomp_metrics[site]
        stats = entry.get('decomposition_stats', {})
        tempo_comp = entry.get('tempo_comparison', {})

        for tempo_type in ['TEMPO Zero-Shot', 'TEMPO Fine-Tuned']:
            if tempo_type not in tempo_comp:
                continue
            c = tempo_comp[tempo_type]
            decomp_rows.append({
                'Site': site,
                'Model': tempo_type,
                'Trend r': c.get('trend', {}).get('correlation', np.nan),
                'Seasonal r': c.get('seasonal', {}).get('correlation', np.nan),
                'Residual r': c.get('resid', {}).get('correlation', np.nan),
                'Trend Var %': stats.get('trend_pct', np.nan),
                'Seasonal Var %': stats.get('seasonal_pct', np.nan),
                'Residual Var %': stats.get('residual_pct', np.nan),
            })

    decomp_df = pd.DataFrame(decomp_rows)

    if not decomp_df.empty:
        print('=== STL Component Correlations ===\n')
        display(decomp_df)
    else:
        print('No TEMPO comparison data found in decomposition metrics.')

In [ ]:
if decomp_metrics and not decomp_df.empty:
    avail_sites = decomp_df['Site'].unique()
    n_sites = len(avail_sites)

    fig, axes = plt.subplots(
        1, max(n_sites, 1),
        figsize=(8 * max(n_sites, 1), 6),
        squeeze=False,
    )

    comp_cols = ['Trend r', 'Seasonal r', 'Residual r']
    x = np.arange(len(comp_cols))
    w = 0.35

    for ax_idx, site in enumerate(avail_sites):
        ax = axes[0, ax_idx]
        site_data = decomp_df[decomp_df['Site'] == site]

        zs_row = site_data[site_data['Model'] == 'TEMPO Zero-Shot']
        ft_row = site_data[site_data['Model'] == 'TEMPO Fine-Tuned']

        if not zs_row.empty:
            vals = zs_row[comp_cols].values[0]
            bars1 = ax.bar(
                x - w / 2, vals, w,
                label='Zero-Shot', color=MODEL_COLORS['TEMPO Zero-Shot'], alpha=0.85,
            )
            for i, v in enumerate(vals):
                ax.text(i - w / 2, v + 0.02, f'{v:.3f}',
                        ha='center', fontsize=9, fontweight='bold')

        if not ft_row.empty:
            vals = ft_row[comp_cols].values[0]
            bars2 = ax.bar(
                x + w / 2, vals, w,
                label='Fine-Tuned', color=MODEL_COLORS['TEMPO Fine-Tuned'], alpha=0.85,
            )
            for i, v in enumerate(vals):
                ax.text(i + w / 2, v + 0.02, f'{v:.3f}',
                        ha='center', fontsize=9, fontweight='bold')

        # Variance annotation below x-axis
        var_cols = ['Trend Var %', 'Seasonal Var %', 'Residual Var %']
        var_vals = site_data[var_cols].values[0]
        for i, vp in enumerate(var_vals):
            ax.text(i, -0.12, f'({vp:.1f}% var)',
                    ha='center', fontsize=8, style='italic', color='gray')

        ax.set_ylabel('Correlation', fontsize=11, fontweight='bold')
        ax.set_title(
            f"{site} ({SITE_INFO[site]['ecosystem']})",
            fontsize=12, fontweight='bold',
        )
        ax.set_xticks(x)
        ax.set_xticklabels(['Trend', 'Seasonal', 'Residual'])
        ax.set_ylim(-0.15, 1.05)
        ax.axhline(0, color='black', lw=0.8)
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3, axis='y')

    plt.suptitle(
        'TEMPO STL Component Correlation Analysis',
        fontsize=14, fontweight='bold',
    )
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'tempo_component_correlations.png', dpi=300, bbox_inches='tight')
    plt.savefig(FIG_DIR / 'tempo_component_correlations.pdf', bbox_inches='tight')
    plt.show()
    print('Figure saved: tempo_component_correlations.png')

## 11. Ecological Validation Summary

Reporting the ecological pattern checks from the KGML decomposition analysis.

In [ ]:
if decomp_metrics:
    for site in SITES:
        if site not in decomp_metrics:
            print(f'{site}: no decomposition data (run --site {site})')
            continue

        entry = decomp_metrics[site]
        stats = entry.get('decomposition_stats', {})
        val = entry.get('validation', {})
        checks = val.get('checks', {})

        print(f"\n{'='*55}")
        print(f"{site} ({SITE_INFO[site]['ecosystem']})")
        print(f"{'='*55}")
        print(f"Timesteps: {stats.get('n_timesteps', '?'):,}")
        print(f"Variance decomposition:")
        print(f"  Trend:    {stats.get('trend_pct', 0):5.1f}%")
        print(f"  Seasonal: {stats.get('seasonal_pct', 0):5.1f}%")
        print(f"  Residual: {stats.get('residual_pct', 0):5.1f}%")
        print(f"\nEcological checks ({val.get('checks_passed', '?')}/{val.get('checks_total', '?')} passed):")

        for name, check in checks.items():
            status = 'PASS' if check.get('pass') else 'FAIL'
            print(f"  [{status}] {name}: {check.get('description', '')}")
else:
    print('Decomposition metrics not available.')

## 12. Unified Results Table (Thesis Table)

A publication-ready table summarising all models across both sites.

In [ ]:
# Pivot into thesis-style table
table_df = comparison_df.copy()
table_df['Model'] = pd.Categorical(
    table_df['Model'], categories=MODEL_ORDER, ordered=True
)
table_df = table_df.sort_values(['Site', 'Model'])

# Format nicely
pivot = table_df.pivot_table(
    index='Model', columns='Site',
    values=['RMSE', 'MAE', 'R\u00b2'],
).round(4)

print('=== Thesis Results Table ===\n')
display(pivot)

# Save as CSV for LaTeX import
csv_path = METRICS_DIR / 'all_models_summary.csv'
flat_rows = []
for _, r in table_df.iterrows():
    flat_rows.append({
        'Model': r['Model'],
        'Site': r['Site'],
        'RMSE': f"{r['RMSE']:.4f}",
        'MAE': f"{r['MAE']:.4f}",
        'R2': f"{r['R\u00b2']:.4f}",
    })
pd.DataFrame(flat_rows).to_csv(csv_path, index=False)
print(f'\nSaved: {csv_path.name}')

## 13. Key Findings

### Zero-Shot Performance

| Site | Ecosystem | R\u00b2 | Interpretation |
|------|-----------|------|----------------|
| **UK-AMo** | Peatland | 0.384 | Weak -- respiration-dominated signal hard to capture from general pre-training |
| **SE-Htm** | Forest | 0.752 | Strong -- forest diurnal/seasonal patterns transfer well from pre-training |

### Fine-Tuning Impact

| Site | RMSE change | R\u00b2 change | Interpretation |
|------|-------------|--------------|----------------|
| **UK-AMo** | -19.3% | +0.215 | Fine-tuning successfully learned wetland-specific patterns |
| **SE-Htm** | +4.7% | -0.024 | Negative transfer -- wetland-dominated training set hurt forest performance |

### STL Component Analysis (KGML)

- **Seasonal** (78% of variance): r = 0.73-0.77 -- TEMPO's strongest alignment, confirms seasonal pattern learning
- **Trend** (9% of variance): r = 0.17-0.33 -- weaker capture of long-term drift
- **Residual** (17% of variance): r \u2248 0.04-0.05 -- expected poor performance (stochastic weather noise)

### Novel Contributions

1. **First application** of TEMPO foundation model to carbon flux forecasting
2. **Demonstrated** zero-shot capability: R\u00b2 = 0.75 on forest with no domain training
3. **Revealed** ecosystem-dependent fine-tuning trade-off: wetland gains vs forest degradation
4. **Validated** knowledge-guided decomposition: TEMPO implicitly captures seasonal structure

### Implications

- **Wetlands**: Foundation models alone insufficient; fine-tuning provides +56% relative R\u00b2 improvement
- **Forests**: Pre-trained model already captures dynamics well; fine-tuning can cause negative transfer
- **Future work**: Ecosystem-stratified fine-tuning or frozen-encoder strategies to avoid negative transfer

In [ ]:
print('=' * 70)
print('TEMPO ANALYSIS COMPLETE')
print('=' * 70)
print('\nFigures generated:')
fig_files = sorted(set(
    list(FIG_DIR.glob('*tempo*.png')) + list(FIG_DIR.glob('all_models*.png'))
))
for p in fig_files:
    print(f'  {p.name}')
print(f'\nAll figures saved to: {FIG_DIR.resolve()}')
print('\n=== THESIS-READY RESULTS ===')
print('All analysis notebooks complete!')
print('  01_data_exploration.ipynb    - Data characterisation')
print('  02_baseline_experiments.ipynb - RF / XGBoost / LSTM baselines')
print('  03_tempo_analysis.ipynb      - TEMPO zero-shot, fine-tuned, KGML')
print('\nNext step: Write your Results chapter using these figures!')